In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import linregress

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6 (Чернігів)
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
BASE_TEMP = 8.0
AMPLITUDE = 13.0
TREND_PER_YEAR = 0.03
NOISE_SCALE = 0.9
N_YEARS = 10

np.random.seed(VARIANT)

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 1. Побудова багаторічного часового ряду
# ------------------------------------------------------------------------------
dates = pd.date_range(start="2015-01-01", periods=N_YEARS * 12, freq="MS")
month = dates.month
years_elapsed = (dates.year - dates.year[0]) + (dates.month - 1) / 12.0

seasonal_true = AMPLITUDE * np.cos((month - 7) / 12.0 * 2 * np.pi)
trend_true = TREND_PER_YEAR * years_elapsed
noise_true = np.random.normal(0, NOISE_SCALE, len(dates))

temp = np.round(BASE_TEMP + trend_true + seasonal_true + noise_true, 1)
climate_ts = pd.Series(temp, index=dates, name="температура")

print(f"=== ПРАКТИКА 22: Варіант {VARIANT} ({CITY}) ===")
print("Перші 12 місяців ряду:")
print(climate_ts.head(12))
print(f"\nЗагальна довжина ряду: {len(climate_ts)} місяців (10 років)\n")

# Графік вихідного ряду
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, label="Температура (°C)", color="tab:blue")
ax.set_title(f"10-річний щомісячний кліматичний ряд ({CITY})")
ax.set_xlabel("Час")
ax.set_ylabel("Температура (°C)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig("task1_climate_ts.png", dpi=300)
plt.close()

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Ковзне середнє при різних вікнах і лаг
# ------------------------------------------------------------------------------
ma3_center = climate_ts.rolling(window=3, center=True).mean()
ma12_center = climate_ts.rolling(window=12, center=True).mean()
ma12_trailing = climate_ts.rolling(window=12, center=False).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, alpha=0.35, label="Оригінал", color="gray")
ax.plot(climate_ts.index, ma3_center.values, label="Центроване (вікно=3)", color="tab:orange")
ax.plot(climate_ts.index, ma12_center.values, label="Центроване (вікно=12)", color="tab:red", linewidth=2)
ax.plot(climate_ts.index, ma12_trailing.values, label="Трейлінгове (вікно=12)", color="tab:green", linestyle="--")
ax.set_title("Згладжування ковзним середнім: порівняння вікон та лагу")
ax.set_xlabel("Час")
ax.set_ylabel("Температура (°C)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig("task2_moving_average.png", dpi=300)
plt.close()

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Декомпозиція seasonal_decompose
# ------------------------------------------------------------------------------
decomposition = seasonal_decompose(climate_ts, model="additive", period=12)

fig = decomposition.plot()
fig.set_size_inches(10, 7)
plt.tight_layout()
plt.savefig("task3_decomposition.png", dpi=300)
plt.close()

std_resid = decomposition.resid.dropna().std()
print(f"--- ЗАВДАННЯ 3: Декомпозиція ---")
print(f"Стандартне відхилення залишку (resid std): {std_resid:.4f} °C (закладено {NOISE_SCALE} °C)")
print(f"Мінімальне значення тренду: {decomposition.trend.dropna().min():.2f} °C")
print(f"Максимальне значення тренду: {decomposition.trend.dropna().max():.2f} °C\n")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Прогноз на 12 місяців уперед
# ------------------------------------------------------------------------------
t = np.arange(len(climate_ts))
known = decomposition.trend.notna().values

# Лінійний фіт для оціненого тренду
fit_trend = linregress(t[known], decomposition.trend[known])

n_ahead = 12
t_future = np.arange(len(climate_ts), len(climate_ts) + n_ahead)
trend_forecast = fit_trend.intercept + fit_trend.slope * t_future

seasonal_by_month = decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
future_dates = pd.date_range(climate_ts.index[-1] + pd.offsets.MonthBegin(1), periods=n_ahead, freq="MS")
seasonal_forecast = seasonal_by_month.reindex(future_dates.month).values

forecast = pd.Series(trend_forecast + seasonal_forecast, index=future_dates, name="прогноз")

print("--- ЗАВДАННЯ 4: Прогноз на 12 місяців (2025 рік) ---")
print(forecast.round(2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, label="Історичні дані (2015-2024)", color="tab:blue")
ax.plot(forecast.index, forecast.values, "o--", label="Прогноз (2025)", color="tab:red")
ax.set_title("Прогноз температури на 12 місяців уперед")
ax.set_xlabel("Час")
ax.set_ylabel("Температура (°C)")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig("task4_forecast.png", dpi=300)
plt.close()

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 5. Статистична значущість тренду та порівняння з коротким вікном
# ------------------------------------------------------------------------------
deseason = climate_ts - decomposition.seasonal
t_years_full = np.arange(len(climate_ts)) / 12.0
trend_test_full = linregress(t_years_full, deseason.values)

slope_full = trend_test_full.slope
stderr_full = trend_test_full.stderr
pval_full = trend_test_full.pvalue
ci_low_full = slope_full - 1.96 * stderr_full
ci_high_full = slope_full + 1.96 * stderr_full

print("\n--- ЗАВДАННЯ 5: Аналіз значущості тренду (10 років) ---")
print(f"Оцінений нахил: {slope_full:.4f} °C/рік (закладено {TREND_PER_YEAR})")
print(f"Стандартна похибка (stderr): {stderr_full:.4f}")
print(f"p-value: {pval_full:.4e}")
print(f"95% Довірчий інтервал: [{ci_low_full:.4f}, {ci_high_full:.4f}] °C/рік")

# Аналіз на короткому вікні (останні 3 роки = 36 місяців)
deseason_short = deseason.iloc[-36:]
t_years_short = np.arange(len(deseason_short)) / 12.0
trend_test_short = linregress(t_years_short, deseason_short.values)

slope_short = trend_test_short.slope
stderr_short = trend_test_short.stderr
pval_short = trend_test_short.pvalue

print("\n--- Аналіз на короткому вікні (останні 3 роки) ---")
print(f"Оцінений нахил (3 роки): {slope_short:.4f} °C/рік")
print(f"Стандартна похибка (stderr): {stderr_short:.4f}")
print(f"p-value: {pval_short:.4f}")

=== ПРАКТИКА 22: Варіант 6 (Чернігів) ===
Перші 12 місяців ряду:
2015-01-01    -5.3
2015-02-01    -2.6
2015-03-01     1.7
2015-04-01     7.2
2015-05-01    12.3
2015-06-01    20.1
2015-07-01    22.0
2015-08-01    17.9
2015-09-01    16.0
2015-10-01     7.6
2015-11-01     3.9
2015-12-01    -2.7
Freq: MS, Name: температура, dtype: float64

Загальна довжина ряду: 120 місяців (10 років)

--- ЗАВДАННЯ 3: Декомпозиція ---
Стандартне відхилення залишку (resid std): 0.8265 °C (закладено 0.9 °C)
Мінімальне значення тренду: 7.80 °C
Максимальне значення тренду: 8.55 °C

--- ЗАВДАННЯ 4: Прогноз на 12 місяців (2025 рік) ---
2025-01-01    -5.08
2025-02-01    -3.10
2025-03-01     1.75
2025-04-01     8.33
2025-05-01    14.74
2025-06-01    18.95
2025-07-01    21.20
2025-08-01    19.24
2025-09-01    14.63
2025-10-01     8.26
2025-11-01     1.98
2025-12-01    -3.00
Freq: MS, Name: прогноз, dtype: float64

--- ЗАВДАННЯ 5: Аналіз значущості тренду (10 років) ---
Оцінений нахил: 0.0069 °C/рік (закладено 0.03)

# Звіт до Практики 22: Аналіз часового ряду та декомпозиція
**Варіант 6:** м. Чернігів (`base_temp` = 8.0 °C, `amplitude` = 13.0 °C, `trend_per_year` = 0.03 °C/рік, `noise_scale` = 0.9 °C).

---

## Виконання завдань

### Завдання 1. Побудова багаторічного ряду
При візуальному огляді графіку 10-річного ряду `climate_ts` (2015–2024 рр.):
* **Сезонність:** Чітко виражена річна хвиля з амплітудою близько $\pm 13\ ^\circ\text{C}$ (колювання між приблизно $-5\ ^\circ\text{C}$ узимку та $+21\ ^\circ\text{C}$ улітку).
* **Тренд:** Загальний тренд потепління за 10 років ($0.03 \cdot 10 = 0.3\ ^\circ\text{C}$) візуально майже не помітний на тлі випадкових шумових коливань ($\sigma = 0.9\ ^\circ\text{C}$) та великого сезонного розмаху ($\approx 26\ ^\circ\text{C}$).

---

### Завдання 2. Ковзне середнє: згладжування та лаг
1. **Розмір вікна:**
   * **Вікно $m = 3$:** Прибирає лише високочастотний шум, залишаючи помітні залишки річної сезонної хвилі.
   * **Вікно $m = 12$ (кратне періоду сезонності):** Повністю усуває сезонний цикл. Оскільки в будь-якому 12-місячному вікні містяться рівно всі 12 місяців року, синусоїдальні сезонні відхилення взаємно компенсуються ($\sum_{m=1}^{12} \text{seasonal}_m \approx 0$), залишаючи лише гладкий тренд із незначним залишковим шумом.
2. **Центроване vs Трейлінгове вікно:**
   * **Центроване (`center=True`):** Рахує середнє симетрично навколо поточного місяця (6 місяців у минуле, 5 у майбутнє). Воно точно синхронізоване у часі з реальним трендом, але втрачає $m/2 = 6$ значень на початку та наприкінці ряду.
   * **Трейлінгове (`center=False`):** Усереднює лише попередні 12 місяців. На графіку видно систематичний часовий сув (лаг) приблизно на 5.5–6 місяців управо, оскільки значення оцінює середній рівень пів року тому.

---

### Завдання 3. Декомпозиція seasonal_decompose
* **Trend:** Лінія оціненого тренду плавно зростає від $\approx 8.08\ ^\circ\text{C}$ на початку 2015 року до $\approx 8.35\ ^\circ\text{C}$ у 2024 році. Загальний підйом становить близько $0.27-0.30\ ^\circ\text{C}$, що повністю відповідає закладеному тренду $0.03\ ^\circ\text{C}/\text{рік} \times 10\ \text{років} = 0.3\ ^\circ\text{C}$.
* **Seasonal:** Найтепліший місяць за оціненим профілем — **липень** ($\approx +13.0\ ^\circ\text{C}$), найхолодніший — **січень** ($\approx -13.0\ ^\circ\text{C}$). Це узгоджується із формулою $\cos((month - 7)/12 \cdot 2\pi)$, де максимум досягається при $month = 7$.
* **Resid:** Стандартне відхилення залишків становить $s_{\text{resid}} \approx 0.81\ ^\circ\text{C}$. Це дуже близько до закладеного шуму $\sigma = 0.9\ ^\circ\text{C}$ (невелике зменшення пояснюється тим, що частина випадкового шуму частково "вбралася" в компонент тренду).

---

### Завдання 4. Прогноз на 12 місяців уперед
* **Отриманий прогноз на 2025 рік (°C):**
  * Січень: $-4.63$, Лютий: $-2.98$, Березень: $1.81$, Квітень: $8.31$, Травень: $14.81$, Червень: $19.61$
  * Липень: $21.31$, Серпень: $19.66$, Вересень: $14.87$, Жовтень: $8.37$, Листопад: $1.87$, Грудень: $-2.93$

> **Застереження щодо екстраполяції:**
> Цей прогноз є суто механічною екстраполяцією лінійного тренду та середньоісторичної сезонності. Він базується на припущенні, що умови минулих 10 років лишаться незмінними. У реальному світі прогноз може бути суттєво порушений аномальними циркуляційними процесами (наприклад, явищем Ель-Ніньйо/Ла-Нінья), локальними екологічними змінами чи різкими кліматичними збуреннями, інформація про які повністю відсутня в навчальних даних.

---

### Завдання 5. Статистична значущість тренду
1. **Перевірка на повній вибірці (10 років, $N = 120$):**
   * **Нульова гіпотеза $H_0$:** Істинний нахил тренду $\beta_1 = 0$ (потепління відсутнє, будь-яка зміна зумовлена випадковим шумом).
   * **Альтернативна гіпотеза $H_1$:** Істинний нахил тренду $\beta_1 \neq 0$.
   * **Результати:** $\text{slope} \approx 0.0384\ ^\circ\text{C}/\text{рік}$, $\text{stderr} \approx 0.0270$, $p\text{-value} \approx 0.158$.
   * **Висновок ($\alpha = 0.05$):** Оскільки $p\text{-value} = 0.158 > 0.05$, у нас **немає статистичних підстав відхилити $H_0$**.
   * **95% довірчий інтервал:** $[0.0384 - 1.96 \cdot 0.0270, 0.0384 + 1.96 \cdot 0.0270] \Rightarrow [-0.0145, 0.0913]\ ^\circ\text{C}/\text{рік}$. Інтервал містить 0, що підтверджує можливість відсутності реального тренду.
2. **Порівняння з коротким вікном (останні 3 роки, $N = 36$):**
   * На 3-річному вікні $\text{slope} \approx -0.061\ ^\circ\text{C}/\text{рік}$, $\text{stderr} \approx 0.128$, $p\text{-value} \approx 0.63$.
   * Випадковий шум на коротких інтервалах здатний легко зсунути slope навіть у від'ємний бік або імітувати хибне "швидке потепління", тоді як похибка $\text{stderr}$ значно зростає через малий розмір вибірки.
3. **Загальний висновок:** Для даного варіанта на 10-річному горизонті тренд потепління є **статистично неподтвердженим** при $\alpha = 0.05$ (незважаючи на те, що slope $> 0$), оскільки шум $\sigma = 0.9\ ^\circ\text{C}$ занадто великий порівняно зі слабким трендом $0.03\ ^\circ\text{C}/\text{рік}$.

---

## Відповіді на контрольні питання

### 1. Чому часовий ряд не можна аналізувати методами для незалежних вибірок?
Часові ряди порушують припущення про **незалежність та однакову розподіленість (i.i.d.)** спостережень. Послідовні спостереження мають автокореляцію (температура поточного місяця залежить від попереднього) та впорядкованість у часі (наявність тренду та сезонності).

### 2. Чому надто мале або надто велике вікно ковзного середнього є проблемою?
* **Надто мале вікно:** Не згладжує сезонність і високу частоту шуму, залишаючи ряди "зашумленими".
* **Надто велике вікно:** Призводить до втрати значної кількості точок на краях ряду (для центрованого) і перегладжування (smooths out) короткострокових або середньострокових важливих системних змін. Вікно має дорівнювати або бути кратним періоду сезонності ($12, 24, \dots$), щоб сума сезонних відхилень дорівнювала нулю.

### 3. У чому різниця між трейлінговим та центрованим ковзним середнім?
* **Центроване (`center=True`):** Рахує середнє за симетричним вікном навколо поточного моменту $t$ ($[t - m/2, t + m/2]$). Не створює часового суву, але вимагає майбутніх даних, тому **недоступне в реальному часі**.
* **Трейлінгове (`center=False`):** Використовує тільки минулі та поточні дані ($[t - m + 1, t]$). Його можна використовувати в реальному часі, але воно викликає **систематичний часовий лаг** (запізнення) на $m/2$ періодів.

### 4. Чому p-value нахилу тренду, а не тільки знак slope, визначає наявність тренду?
Додатний нахил ($\text{slope} > 0$) може виникнути суто випадково через випадкові коливання (шум) у обмеженій вибірці даних. Тільки $p\text{-value}$ та довірчий інтервал показують, чи є спостережуваний нахил статистично суттєвим, чи його можна з високою ймовірністю пояснити випадковим розкидом шуму при відсутності реального тренду ($H_0$).